# The Fairness Ledger: Safewater Lines Employee Survey Dashboard

TISS live industry project: *From Employee Voices to Organisational Impact*.

**How to run:** set `SOURCE_FILE` in Cell 1 to your Excel file, then run every cell from top to bottom (Kernel > Restart & Run All). The last cell writes `fairness_ledger_dashboard.html` and opens it in your browser.

**What it shows:** a hero balance plus five charts. Every chart has its own filters.

**What it needs:** `pandas`, `numpy`, `scipy`, `openpyxl`, and `plotly` (plotly is only used to embed its JavaScript into the page so the dashboard works with no internet).

**Data rule:** only the `SWL - MASTERDATA` sheet is read. Every number is computed from that sheet.

## Cell 0: Install (run once, then comment out again)

In [ ]:
# !pip install pandas numpy scipy openpyxl plotly

## Cell 1: Configuration
Every setting lives here: file paths, the answer-to-score map, the six dimensions, short labels for all 30 questions, and the comment themes. Nothing further down has magic numbers.

In [1]:
import json
import re
import webbrowser
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# ---- CHANGE THIS to your file. Keep the r before the quote on Windows. ----
SOURCE_FILE = r"C:\Users\bhogi\Downloads\Your Voice- Responses 21 OCT - FINAL.xlsx"
OUTPUT_FILE = r"C:\Users\bhogi\OneDrive\Desktop\Yug.html"
OPEN_IN_BROWSER = True
# Fonts are downloaded once at build time and embedded, so the page keeps
# its look with no internet. If the download fails, it links to Google instead.
FONTS_URL = ("https://fonts.googleapis.com/css2?family=Archivo+Black"
             "&family=Figtree:wght@400;500;600;700;800"
             "&family=Fraunces:wght@300;400;500&display=swap")
# ---------------------------------------------------------------------------

SHEET_NAME = "SWL - MASTERDATA"
HEADER_ROW = 1   # row 2 in Excel holds the real question titles

# Survey answer text -> score. Text is lowercased and trimmed first.
LIKERT_MAP = {
    "always": 5, "almost always": 5,
    "often": 4, "usually": 4,
    "sometimes": 3,
    "rarely": 2,
    "almost never": 1,
}

# Item positions 0..29 = the 30 Likert columns (sheet columns 6..35).
DIMENSIONS = {
    "Credibility":        [0, 1, 2, 3, 4],
    "Respect":            [5, 6, 7, 8, 9],
    "Fairness":           [10, 11, 12, 13, 14],
    "Pride":              [15, 18, 19],      # 16 and 17 deliberately left out
    "Camaraderie":        [20, 21, 22, 23, 24],
    "Overall Experience": [26, 27, 28, 29],
}
OUTCOMES = {"stay": 17, "recommend": 25}   # never averaged into a dimension

ITEM_SHORT = [
    "Leadership is open and honest", "Leaders keep their word",
    "Confidence in direction", "Timely updates on change",
    "Leaders act on feedback", "Manager supports me",
    "Resources and recognition", "Safe to speak up",
    "Effort is appreciated", "Dignity regardless of role",
    "Rewards are fair", "Equal chance to grow",
    "Decisions without favouritism", "Workload shared fairly",
    "Understand how decisions are made", "Proud to say I work here",
    "See how my work matters", "See myself here in 3-5 years",
    "Positive impact on customers", "Go beyond when needed",
    "Team supports each other", "Teams work well together",
    "I belong here", "Can rely on other teams",
    "Successes are celebrated", "Would recommend as a workplace",
    "Room to learn and grow", "Have what I need to perform",
    "Satisfied overall", "Meaningful work",
]

# The survey asked tenure in unequal bands (0-3, 3-5, 5-10, 10-15, 15+). Exact years were not
# collected, so the bands cannot be re-cut. Merging 0-3 and 3-5 gives equal 5-year buckets;
# 15+ stays open-ended because that is how the survey asked it.
TENURE_MERGE = {"0 - 3": "0 - 5", "3 - 5": "0 - 5"}
TENURE_ORDER = ["0 - 5", "5 - 10", "10 - 15", "15 & Above"]
TENURE_LABELS = {"0 - 5": "0-5 yrs", "5 - 10": "5-10 yrs",
                 "10 - 15": "10-15 yrs", "15 & Above": "15+ yrs"}

# Comment themes for Chart 4. Keyword matching is approximate by nature.
THEMES = [
    ("Communication and transparency", "Process",
     [r"communicat\w*", r"transparen\w*", r"inform\w*", r"update[sd]?",
      r"honest\w*", r"clarity"]),
    ("Recognition and appreciation", "Process",
     [r"recogni[sz]\w*", r"appreciat\w*", r"acknowledg\w*", r"reward\w*"]),
    ("Growth, career and training", "Process",
     [r"growth", r"career\w*", r"train\w*", r"develop\w*", r"promot\w*",
      r"upskill\w*", r"mentor\w*", r"learn\w*"]),
    ("Respect, listening and voice", "Process",
     [r"respect\w*", r"listen\w*", r"voice", r"heard", r"dignity"]),
    ("Leadership and management", "Process",
     [r"leader\w*", r"management", r"manager\w*"]),
    ("Teamwork and coordination", "Process",
     [r"teamwork\w*", r"collaborat\w*", r"coordinat\w*", r"silo\w*"]),
    ("Fairness and favouritism", "Process",
     [r"fair\w*", r"favorit\w*", r"favourit\w*", r"bias\w*", r"partial\w*"]),
    ("Workload and balance", "Process",
     [r"workload\w*", r"overtime", r"burnout", r"balance"]),
    ("Pay and compensation", "Budget",
     [r"salary\w*", r"pay", r"compensat\w*", r"increment\w*", r"bonus\w*",
      r"wage\w*"]),
    ("Resources, tools and systems", "Budget",
     [r"resources?", r"tools?", r"systems?", r"software", r"equipment",
      r"infrastructure"]),
]

print("Cell 1 done: configuration loaded.")

Cell 1 done: configuration loaded.


## Cell 2: Load the data
Reads the one approved sheet and nothing else in the workbook.

In [2]:
if not Path(SOURCE_FILE).exists():
    raise FileNotFoundError(
        f"Could not find {SOURCE_FILE!r}. Set SOURCE_FILE in Cell 1 to the "
        "full path of your Excel file, with r before the opening quote."
    )

raw_df = pd.read_excel(r"C:\Users\bhogi\OneDrive\Desktop\Yug\SWL\Your Voice- Responses 21 OCT - FINAL.xlsx", sheet_name="SWL - MASTERDATA", header=HEADER_ROW)
raw_df.columns = [str(c).strip() for c in raw_df.columns]
N_TOTAL = len(raw_df)

print(f"Cell 2 done: {N_TOTAL} respondents loaded from '{SHEET_NAME}' only.")

Cell 2 done: 327 respondents loaded from 'SWL - MASTERDATA' only.


## Cell 3: Clean and score
Splits the sheet into demographics, 1-5 scores and comments. Names are dropped here and never used again. Stops with a clear error if any answer text does not match the score map.

In [3]:
demo = raw_df.iloc[:, 0:6].copy()
demo.columns = ["location", "designation", "department", "tenure", "gender", "name"]
demo = demo.drop(columns=["name", "designation"])
for col in demo.columns:
    demo[col] = demo[col].fillna("Not stated").astype(str).str.strip()
demo["tenure"] = demo["tenure"].replace(TENURE_MERGE)   # equal 5-year buckets

QUESTIONS = [str(c).strip() for c in raw_df.columns[6:36]]


def to_score(value):
    if pd.isna(value):
        return np.nan
    return LIKERT_MAP.get(str(value).strip().lower(), np.nan)


scores = raw_df.iloc[:, 6:36].apply(lambda col: col.map(to_score))
scores.columns = list(range(30))

unmatched = int(scores.isna().sum().sum())
if unmatched:
    raise ValueError(f"{unmatched} answers did not match LIKERT_MAP. Check the raw text.")
scores = scores.astype(int)

open_text = raw_df.iloc[:, 36:43].fillna("").astype(str)

dim_scores = pd.DataFrame({name: scores[idx].mean(axis=1)
                           for name, idx in DIMENSIONS.items()})

print(f"Cell 3 done: {scores.size:,} answers scored, 0 unmatched.")

Cell 3 done: 9,810 answers scored, 0 unmatched.


## Cell 4: Tag comment themes per person
For each person, a 1 or 0 per theme: did any of their seven comments mention it. Counting people rather than raw keyword hits stops one long comment from outweighing everyone else. Only these flags go into the dashboard, never the comment text.

In [4]:
person_text = open_text.agg(" ".join, axis=1).str.lower()
theme_flags = np.zeros((N_TOTAL, len(THEMES)), dtype=int)
for j, (_, _, patterns) in enumerate(THEMES):
    rx = re.compile(r"\b(?:" + "|".join(patterns) + r")\b")
    theme_flags[:, j] = person_text.apply(lambda s: 1 if rx.search(s) else 0).values

print("Cell 4 done: share of people mentioning each theme")
for j, (name, need, _) in enumerate(THEMES):
    print(f"  {name:32s} {100 * theme_flags[:, j].mean():5.1f}%  ({need})")

Cell 4 done: share of people mentioning each theme
  Communication and transparency    57.2%  (Process)
  Recognition and appreciation      45.0%  (Process)
  Growth, career and training       44.6%  (Process)
  Respect, listening and voice      55.0%  (Process)
  Leadership and management         33.0%  (Process)
  Teamwork and coordination         34.6%  (Process)
  Fairness and favouritism          25.4%  (Process)
  Workload and balance              11.9%  (Process)
  Pay and compensation               8.0%  (Budget)
  Resources, tools and systems       6.4%  (Budget)


## Cell 5: Company-wide statistics
The tests behind the claims on the page. Run once on all respondents. The dashboard shows these markers only when no filter is applied, because that is the data they were run on.

In [6]:
dimension_means = {d: round(float(dim_scores[d].mean()), 2) for d in DIMENSIONS}

corr_r, corr_p = stats.pearsonr(dim_scores["Fairness"], scores[OUTCOMES["recommend"]])

cs_mask = demo["department"] == "Customer Service"
cs_t, cs_p = stats.ttest_ind(dim_scores.loc[cs_mask, "Fairness"],
                             dim_scores.loc[~cs_mask, "Fairness"], equal_var=False)

tenure_steps = []
for a, b in zip(TENURE_ORDER[:-1], TENURE_ORDER[1:]):
    g1 = dim_scores.loc[demo["tenure"] == a, "Fairness"]
    g2 = dim_scores.loc[demo["tenure"] == b, "Fairness"]
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    tenure_steps.append({"from": a, "to": b,
                         "change": round(float(g2.mean() - g1.mean()), 2),
                         "p": round(float(p), 3), "significant": bool(p < 0.05)})

FULL_STATS = {
    "dimension_means": dimension_means,
    "corr_r": round(float(corr_r), 2),
    "corr_p": float(corr_p),
    "cs_fairness": round(float(dim_scores.loc[cs_mask, "Fairness"].mean()), 2),
    "rest_fairness": round(float(dim_scores.loc[~cs_mask, "Fairness"].mean()), 2),
    "cs_p": round(float(cs_p), 4),
    "tenure_steps": tenure_steps,
}

print("Cell 5 done")
for d, v in dimension_means.items():
    print(f"  {d:20s} {v}")
print(f"  Fairness vs recommend: r = {FULL_STATS['corr_r']}")
print(f"  Customer Service {FULL_STATS['cs_fairness']} vs rest "
      f"{FULL_STATS['rest_fairness']} (p = {FULL_STATS['cs_p']})")
for s in tenure_steps:
    print(f"  {s['from']:>10s} -> {s['to']:<10s} {s['change']:+.2f}  p={s['p']}"
          f"  {'significant' if s['significant'] else ''}")

Cell 5 done
  Credibility          4.35
  Respect              4.32
  Fairness             4.17
  Pride                4.58
  Camaraderie          4.35
  Overall Experience   4.47
  Fairness vs recommend: r = 0.67
  Customer Service 3.79 vs rest 4.24 (p = 0.0013)
       0 - 5 -> 5 - 10     +0.14  p=0.242  
      5 - 10 -> 10 - 15    +0.41  p=0.004  significant
     10 - 15 -> 15 & Above -0.47  p=0.001  significant


## Cell 6: Build the dashboard data
One compact record per person: department, tenure, location, gender, their 30 scores and their theme flags. No names and no comment text. The browser does all filtering and maths from this, so every filter combination works, including one person or nobody.

In [7]:
def category_list(series, order=None):
    counts = series.value_counts()
    names = [o for o in (order or []) if o in counts.index]
    names += [v for v in counts.index if v not in names]
    return names, [{"name": v, "n": int(counts[v])} for v in names]


dept_names, DEPTS = category_list(demo["department"])
ten_names, TENURES = category_list(demo["tenure"], TENURE_ORDER)
loc_names, LOCATIONS = category_list(demo["location"])
gen_names, GENDERS = category_list(demo["gender"])
for t in TENURES:
    t["label"] = TENURE_LABELS.get(t["name"], t["name"])

ROWS = []
for i in range(N_TOTAL):
    ROWS.append([
        dept_names.index(demo.at[i, "department"]),
        ten_names.index(demo.at[i, "tenure"]),
        loc_names.index(demo.at[i, "location"]),
        gen_names.index(demo.at[i, "gender"]),
        "".join(str(v) for v in scores.iloc[i].tolist()),
        "".join(str(v) for v in theme_flags[i].tolist()),
    ])

item_dim = {}
for name, idx in DIMENSIONS.items():
    for k in idx:
        item_dim[k] = name
item_dim[16] = "Pride"
for k in OUTCOMES.values():
    item_dim[k] = "Outcome"

PAYLOAD = {
    "total": N_TOTAL,
    "rows": ROWS,
    "depts": DEPTS, "tenures": TENURES, "locations": LOCATIONS, "genders": GENDERS,
    "dimensions": DIMENSIONS,
    "outcomes": OUTCOMES,
    "items": [{"short": ITEM_SHORT[k], "question": QUESTIONS[k], "dim": item_dim[k]}
              for k in range(30)],
    "themes": [{"name": n, "need": need} for n, need, _ in THEMES],
    "full": FULL_STATS,
}

print(f"Cell 6 done: {len(ROWS)} records, {len(DEPTS)} departments, "
      f"{len(LOCATIONS)} locations, {len(GENDERS)} genders.")

Cell 6 done: 327 records, 26 departments, 23 locations, 3 genders.


## Cell 7: Page styles (CSS)
Palette: the original dark brown, cream and tan, with three meaningful colours used consistently. **Rust orange** = debt / risk / unsettled (the accent, used sparingly). **Deep teal** = settled / staying (orange's complement). **Ochre** = pride. Every block below is labelled.

In [8]:
PAGE_CSS = r"""
/* ---------- 1. tokens ---------- */
:root{
  --esp:#2B2017; --esp2:#3A2C20; --bone:#F6ECDA; --sand:#C9B79C;
  --paper:#F4EDE0; --card:#FFFCF6; --ink:#241B12; --muted:#76664F; --line:#E6DCCB; --line2:#D3C6B1;
  --rust:#C8572E; --rust-l:#E07A4F; --teal:#2F6B6B; --teal-l:#7FB3AD; --ochre:#A8833F; --ochre-l:#D4B47A;
  --shout:"Archivo Black","Arial Black",sans-serif;
  --body:"Figtree",system-ui,sans-serif;
  --display:"Fraunces",Georgia,serif;
  --gutter:clamp(1rem,3.4vw,3.4rem);
}
*,*::before,*::after{box-sizing:border-box}
html{scroll-behavior:smooth;background:var(--esp)}
body{margin:0;background:var(--paper);color:var(--ink);font-family:var(--body);font-size:16px;
  line-height:1.5;-webkit-font-smoothing:antialiased}
:focus-visible{outline:2px solid var(--rust);outline-offset:3px}
.num{font-variant-numeric:lining-nums tabular-nums}

/* ---------- 2. hero shell (one full screen) ---------- */
.hero{position:relative;overflow:hidden;height:100svh;min-height:660px;color:var(--bone);
  display:grid;grid-template-rows:auto minmax(0,1fr) auto;gap:clamp(.2rem,1vh,.8rem);
  padding:clamp(1rem,2.4vh,1.8rem) var(--gutter) clamp(1rem,2.4vh,1.7rem);
  background:radial-gradient(52% 58% at 50% 56%,#45321F 0%,transparent 72%),
             radial-gradient(50% 45% at 0% 100%,#1C140D 0%,transparent 65%),var(--esp)}
.hero-top{display:flex;justify-content:space-between;align-items:flex-start;gap:1rem 2rem;
  flex-wrap:wrap;position:relative;z-index:4}
.brand{display:block;font-family:var(--display);font-size:.98rem;color:var(--sand);margin-bottom:.45rem}
.hero h1{font-family:var(--body);font-weight:800;text-transform:uppercase;margin:0;
  font-size:clamp(1.45rem,2.5vw,2.4rem);line-height:1;letter-spacing:-.015em}
.hero h1 em{font-style:normal;color:var(--rust-l)}
.hero-filters{display:flex;flex-wrap:wrap;gap:.5rem;align-items:center;justify-content:flex-end}

/* ---------- 3. stage: big numbers flanking the balance ---------- */
.stage{position:relative;min-height:0;display:grid;grid-template-rows:minmax(0,1fr);
  grid-template-columns:minmax(7.5rem,1fr) auto minmax(7.5rem,1fr);align-items:center;gap:.5rem}
.watermark{position:absolute;inset:0;display:flex;align-items:center;justify-content:center;
  font-family:var(--shout);font-size:min(22vw,36vh);line-height:1;color:transparent;
  -webkit-text-stroke:1.5px rgba(246,236,218,.07);white-space:nowrap;user-select:none;pointer-events:none;z-index:0}
.balance{position:relative;z-index:2;height:100%;width:auto;max-width:100%;overflow:visible;
  filter:drop-shadow(0 34px 50px rgba(0,0,0,.55))}
.flank{position:relative;z-index:3;display:flex;flex-direction:column;gap:.15rem}
.flank.l{align-items:flex-end;text-align:right}
.flank.r{align-items:flex-start;text-align:left}
.fl-name{font-size:.78rem;letter-spacing:.18em;font-weight:800;color:var(--sand)}
.fl-val{font-family:var(--display);font-weight:400;font-size:clamp(2.8rem,6.2vw,6.6rem);line-height:.95;
  font-variant-numeric:lining-nums tabular-nums;transition:color .4s}
.flank.l .fl-val{color:var(--lc,var(--sand))}
.flank.r .fl-val{color:var(--rust-l)}
.fl-sub{font-size:.85rem;color:var(--sand)}

/* the drawing */
.ground line{stroke:var(--sand);stroke-width:1.4;opacity:.3}
.ground polygon{fill:var(--sand);opacity:.15}
.beam{transform-box:view-box;transform-origin:320px 380px;transition:transform 1s cubic-bezier(.34,1.4,.5,1)}
.beam-bar{fill:var(--bone)}
.pivot{fill:var(--esp);stroke:var(--bone);stroke-width:3}
.ghost-col{fill:none;stroke:var(--sand);stroke-width:1;stroke-dasharray:4 5;opacity:.25}
.blk rect,.blk polygon{stroke:var(--esp);stroke-width:1.1;stroke-linejoin:round}
.left.m-fair .f-front{fill:#C9B79C} .left.m-fair .f-top{fill:#E3D7C3} .left.m-fair .f-side{fill:#8E7F68}
.left.m-debt .f-front{fill:#4E8B86} .left.m-debt .f-top{fill:#86B9B3} .left.m-debt .f-side{fill:#2C5B57}
.right .f-front{fill:#C8572E} .right .f-top{fill:#E3825F} .right .f-side{fill:#8B391C}
.stack{cursor:pointer;transition:opacity .25s}
.stack .hit{fill:transparent}
.balance.hovering .stack:not(.on){opacity:.35}
.blk-in{animation:drop .6s cubic-bezier(.2,.9,.25,1) calc(.15s + var(--i) * .08s) both}
@keyframes drop{from{transform:translateY(-300px);opacity:0}to{transform:none;opacity:1}}

/* ---------- 4. hero footer: caption, FAIR / DEBT, link ---------- */
.hero-foot{display:grid;grid-template-columns:1fr auto 1fr;align-items:center;gap:1.2rem;position:relative;z-index:4}
.mode-caption{margin:0;font-size:.86rem;color:var(--sand);max-width:22rem}
.mode-toggle{display:flex;gap:1.25rem}
.mode-btn{font-family:var(--shout);font-size:clamp(1.3rem,2.1vw,2rem);line-height:1;letter-spacing:.03em;
  min-width:6.2em;padding:.5em 1em;border-radius:999px;cursor:pointer;background:transparent;
  border:2px solid rgba(246,236,218,.32);color:rgba(246,236,218,.7);
  transition:background .25s,color .25s,border-color .25s,transform .2s}
.mode-btn:hover{border-color:var(--bone);color:var(--bone);transform:translateY(-2px)}
.mode-btn[aria-pressed="true"]{background:var(--bone);border-color:var(--bone);color:var(--esp)}
.mode-btn[data-mode="debt"][aria-pressed="true"]{background:var(--rust);border-color:var(--rust);color:#fff}
.explore{justify-self:end;color:var(--bone);text-decoration:none;font-weight:700;font-size:.8rem;
  letter-spacing:.1em;text-transform:uppercase;padding-bottom:4px;border-bottom:2px solid var(--rust-l)}

/* ---------- 5. tooltip for the balance ---------- */
.tip{position:fixed;z-index:60;pointer-events:none;max-width:16rem;background:var(--card);color:var(--ink);
  border:1px solid var(--line);border-radius:10px;padding:.65rem .85rem;font-size:.84rem;line-height:1.4;
  box-shadow:0 14px 34px rgba(0,0,0,.3);opacity:0;transform:translateY(6px);transition:opacity .15s,transform .15s}
.tip.on{opacity:1;transform:none}
.tip b{display:block;font-family:var(--display);font-size:1.5rem;font-weight:400;line-height:1.1}

/* ---------- 6. filter pills (hero and every chart): fixed sizes so nothing jumps ---------- */
.controls{display:flex;flex-wrap:wrap;align-items:center;gap:.55rem;margin:1.1rem 0 .5rem;min-height:38px}
.pill-sel{display:inline-flex;align-items:center;gap:.45rem;height:36px;padding:0 .3rem 0 .85rem;
  background:#F2EADD;border:1px solid var(--line);border-radius:999px;cursor:pointer;
  font-size:.66rem;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:var(--muted);line-height:1;
  transition:background-color .2s ease,border-color .2s ease,color .2s ease}
.pill-sel:hover{border-color:var(--line2);background:#EFE5D5}
.pill-sel select{appearance:none;-webkit-appearance:none;border:0;cursor:pointer;height:34px;width:10.5rem;
  font:inherit;font-size:.85rem;font-weight:600;letter-spacing:0;text-transform:none;color:var(--ink);line-height:34px;
  padding:0 1.7rem 0 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;
  background:transparent url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='10' height='6'%3E%3Cpath d='M1 1l4 4 4-4' stroke='%2376664F' stroke-width='1.6' fill='none'/%3E%3C/svg%3E") no-repeat right .6rem center}
.pill-sel select:focus{outline:none}
.pill-sel:focus-within{box-shadow:0 0 0 2px rgba(200,87,46,.35)}
.pill-sel select option{color:#222;background:#fff}
.pill-sel.on{background:#FBEDE4;border-color:rgba(200,87,46,.55)}
.pill-sel.on span{color:var(--rust)}
.hero .pill-sel{background:rgba(246,236,218,.06);border-color:rgba(246,236,218,.18);color:var(--sand)}
.hero .pill-sel:hover{background:rgba(246,236,218,.1)}
.hero .pill-sel select{color:var(--bone);background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='10' height='6'%3E%3Cpath d='M1 1l4 4 4-4' stroke='%23C9B79C' stroke-width='1.6' fill='none'/%3E%3C/svg%3E")}
.hero .pill-sel.on{background:rgba(200,87,46,.16);border-color:rgba(224,122,79,.6)}
.hero .pill-sel.on span{color:var(--rust-l)}
.reset{font:inherit;font-size:.78rem;font-weight:700;color:var(--rust);background:none;border:0;cursor:pointer;
  height:36px;padding:0 .4rem;text-decoration:underline;text-underline-offset:3px;
  opacity:0;visibility:hidden;transition:opacity .2s ease,visibility .2s}
.reset.show{opacity:1;visibility:visible}
.hero .reset{color:var(--rust-l)}
.seg{display:inline-flex;align-items:center;height:36px;background:#F2EADD;border:1px solid var(--line);border-radius:999px;padding:3px;gap:2px}
.seg button{height:28px;border:0;background:transparent;font:inherit;font-size:.8rem;font-weight:700;color:var(--muted);
  padding:0 .9rem;border-radius:999px;cursor:pointer;transition:background-color .2s ease,color .2s ease}
.seg button:hover{color:var(--ink)}
.seg button[aria-pressed="true"]{background:var(--ink);color:var(--bone)}
.ctl-sep{width:1px;height:22px;background:var(--line2);margin:0 .2rem}

/* ---------- 7. page body: overview tiles and chart cards ---------- */
.wrap{max-width:1320px;margin:0 auto;padding:0 var(--gutter)}
.section{padding-top:clamp(1.8rem,4.5vh,3rem)}
.section:last-of-type{padding-bottom:clamp(2.5rem,7vh,4.5rem)}
.s-label{font-size:.7rem;letter-spacing:.15em;text-transform:uppercase;color:var(--muted);font-weight:800;
  border-left:3px solid var(--rust);padding-left:.65rem;margin:0 0 .55rem}
.ov-title{font-family:var(--display);font-weight:400;font-size:clamp(1.6rem,2.7vw,2.4rem);line-height:1.1;margin:0 0 1rem}
.ov-title b{font-weight:500;color:var(--rust)}
.dims{display:grid;grid-template-columns:repeat(7,minmax(0,1fr));gap:.75rem}
.dim{background:var(--card);border:1px solid var(--line);border-top:3px solid var(--dc,var(--line2));
  border-radius:10px;padding:.85rem .95rem .9rem}
.dim .d-name{font-size:.66rem;letter-spacing:.07em;text-transform:uppercase;color:var(--muted);font-weight:800;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.dim .d-val{font-family:var(--display);font-size:clamp(1.7rem,2.4vw,2.3rem);line-height:1.05;margin:.3rem 0 .15rem;color:var(--vc,var(--ink))}
.dim .d-tag{font-size:.74rem;color:var(--muted);font-weight:600}
.dim .d-bar{height:4px;border-radius:2px;background:#F0E7D9;margin-top:.55rem;position:relative;overflow:hidden}
.dim .d-bar i{position:absolute;left:0;top:0;bottom:0;background:var(--dc,var(--line2));border-radius:2px}
.card{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:1.35rem 1.5rem 1rem;
  box-shadow:0 1px 2px rgba(36,27,18,.03),0 8px 28px rgba(36,27,18,.04)}
.card-top{display:flex;justify-content:space-between;align-items:flex-start;gap:1rem}
.c-title{font-family:var(--display);font-weight:400;font-size:clamp(1.35rem,2.2vw,2rem);line-height:1.15;margin:0;max-width:52rem;min-height:2.3em}
.c-title b{font-weight:500;color:var(--rust)}
.c-title i{font-style:normal;font-weight:500;color:var(--teal)}
.c-sub{margin:.35rem 0 0;color:var(--muted);font-size:.88rem}
.c-n{white-space:nowrap;font-size:.8rem;font-weight:700;color:var(--muted);background:#F2EADD;border-radius:999px;padding:.3rem .75rem;min-width:6.5rem;text-align:center}
.plot{width:100%;transition:opacity .18s ease}
.empty{display:none;align-items:center;justify-content:center;margin:0;padding:2rem 1rem;text-align:center;color:var(--muted);font-size:1rem}
.card.is-empty .plot{display:none}
.card.is-empty .quads{visibility:hidden}
.card.is-empty .empty{display:flex}
.quads{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:.6rem;margin:.6rem 0 .1rem}
.quad{border:1px solid var(--line);border-left:3px solid var(--qc,var(--line2));border-radius:9px;padding:.55rem .8rem;background:#FFFCF6}
.quad .q-name{font-size:.68rem;letter-spacing:.07em;text-transform:uppercase;font-weight:800;color:var(--muted)}
.quad .q-val{font-family:var(--display);font-size:1.5rem;line-height:1.1;margin-top:.1rem}
.quad .q-sub{font-size:.76rem;color:var(--muted)}
.quad .up{color:var(--teal)} .quad .down{color:var(--rust)}

footer{background:var(--esp);color:var(--sand);padding:1.6rem var(--gutter);font-size:.82rem}
footer div{max-width:1320px;margin:0 auto}

/* ---------- 7b. definitions grid and chart notes ---------- */
.defs{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:.75rem}
.def{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:.9rem 1.05rem 1rem}
.def.key{border-top:3px solid var(--rust)}
.def h3{margin:0 0 .3rem;font-family:var(--display);font-weight:500;font-size:1.08rem}
.def p{margin:0;font-size:.88rem;line-height:1.5;color:var(--muted)}
.def p b{color:var(--ink);font-weight:700}
.note{margin:.6rem 0 0;font-size:.82rem;color:var(--muted);font-style:italic;min-height:1.2em}

/* ---------- 8. load motion ---------- */
.hero-top,.hero-foot,.flank{animation:fadeup .8s cubic-bezier(.2,.8,.25,1) both}
.flank{animation-delay:.45s} .hero-foot{animation-delay:.3s}
@keyframes fadeup{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:none}}

/* ---------- 9. smaller screens ---------- */
@media (max-width:1150px){ .dims{grid-template-columns:repeat(4,minmax(0,1fr))} }
@media (max-width:820px){
  .hero{height:auto;min-height:0}
  .stage{grid-template-columns:1fr 1fr;grid-template-rows:auto auto}
  .balance{grid-column:1 / -1;grid-row:1;height:auto;width:100%}
  .flank.l{grid-row:2;align-items:flex-start;text-align:left}
  .flank.r{grid-row:2}
  .hero-filters{justify-content:flex-start}
  .hero-foot{grid-template-columns:1fr;justify-items:center;text-align:center}
  .mode-caption{order:2} .explore{order:3;justify-self:center}
  .dims{grid-template-columns:repeat(2,minmax(0,1fr))}
  .quads{grid-template-columns:repeat(2,minmax(0,1fr))}
  .defs{grid-template-columns:1fr}
  .card{padding:1rem}
  .card-top{flex-direction:column}
}
@media (prefers-reduced-motion:reduce){ *,*::before,*::after{animation:none!important;transition:none!important} }
"""

print("Cell 7 done: page styles ready.")

Cell 7 done: page styles ready.


## Cell 8: Browser logic (JavaScript)
(A) decode data, (B) statistics, (C) the reusable per-chart filter pills, (D) the balance, (E) company overview tiles, (F) shared chart styling, (G-K) the five charts, each with its own filters, (L) start-up.

In [9]:
PAGE_JS = r"""
(function(){
"use strict";
const D = window.LEDGER;
const C = {ink:'#241B12', muted:'#76664F', faint:'#BFB29C', stone:'#A8987F', line:'#E6DCCB', card:'#FFFCF6',
           rust:'#C8572E', teal:'#2F6B6B', ochre:'#A8833F', bone:'#F6ECDA', cream:'#FFFCF6'};
/* muted earth family; only Fairness (rust) and Pride (ochre) carry strong colour */
const DIM_COLORS = {'Credibility':'#6E8C8B','Respect':'#9C8B72','Fairness':C.rust,'Pride':C.ochre,
                    'Camaraderie':'#8DA48F','Overall Experience':'#6B5940'};
const DIM_NAMES = Object.keys(D.dimensions);
const DIM_ALPHA = DIM_NAMES.slice().sort();   /* measure lists (dropdowns, legends) run A to Z */
const FAIR_ITEMS = D.dimensions['Fairness'];
const reduce = window.matchMedia('(prefers-reduced-motion: reduce)').matches;
const $ = id => document.getElementById(id);
const esc = s => String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');

/* ---------- A. one record per person ---------- */
const ROWS = D.rows.map(r => {
  const s = r[4].split('').map(Number);
  const o = {dept:r[0], ten:r[1], loc:r[2], gen:r[3], s:s, th:r[5].split('').map(Number), dim:{}};
  DIM_NAMES.forEach(n => { const ix = D.dimensions[n]; o.dim[n] = ix.reduce((a,i) => a + s[i], 0) / ix.length; });
  o.stays = s[D.outcomes.stay] >= 4;
  o.fairUnsettled = FAIR_ITEMS.filter(i => s[i] < 4).length;
  return o;
});

/* ---------- B. statistics ---------- */
const TCRIT = [NaN,12.706,4.303,3.182,2.776,2.571,2.447,2.365,2.306,2.262,2.228,2.201,2.179,2.160,
  2.145,2.131,2.120,2.110,2.101,2.093,2.086,2.080,2.074,2.069,2.064,2.060,2.056,2.052,2.048,2.045,2.042];
const tcrit = df => df < 1 ? NaN : df <= 30 ? TCRIT[df] : df <= 60 ? 2.000 : df <= 120 ? 1.980 : 1.960;
const mean = a => a.length ? a.reduce((x,y) => x + y, 0) / a.length : null;
function summarise(a){
  const n = a.length; if(!n) return {n:0, mean:null, half:null};
  const m = mean(a); if(n < 2) return {n, mean:m, half:null};
  const v = a.reduce((x,y) => x + (y-m)*(y-m), 0) / (n-1);
  if(v === 0) return {n, mean:m, half:null};
  return {n, mean:m, half: tcrit(n-1) * Math.sqrt(v/n)};
}
function pearson(x, y){
  const n = x.length; if(n < 3) return null;
  const mx = mean(x), my = mean(y); let sxy = 0, sxx = 0, syy = 0;
  for(let i = 0; i < n; i++){ const dx = x[i]-mx, dy = y[i]-my; sxy += dx*dy; sxx += dx*dx; syy += dy*dy; }
  return (sxx === 0 || syy === 0) ? null : sxy / Math.sqrt(sxx*syy);
}
const stayPct = rows => rows.length ? 100 * rows.filter(r => r.stays).length / rows.length : null;
const plural = (n, one, many) => n + ' ' + (n === 1 ? one : many);
const pts = d => (d > 0 ? '+' : '') + Math.round(d) + ' pts';
const wrap = (s, w) => { const out = []; let line = ''; String(s).split(' ').forEach(word => {
  if((line + ' ' + word).trim().length > w){ out.push(line.trim()); line = word; } else line += ' ' + word; });
  out.push(line.trim()); return out.join('<br>'); };

/* company-wide benchmarks, used as reference points across the charts */
const COMPANY = {n:ROWS.length, stay:stayPct(ROWS), dims:{}};
DIM_NAMES.forEach(n => COMPANY.dims[n] = mean(ROWS.map(r => r.dim[n])));
COMPANY.unsettled = 100 * ROWS.reduce((a,r) => a + r.fairUnsettled, 0) / (ROWS.length * FAIR_ITEMS.length);

/* ---------- C. per-chart filter pills ---------- */
const FDEF = {
  dept:{label:'Dept', all:'All departments', list:D.depts},
  ten:{label:'Tenure', all:'All tenure', list:D.tenures},
  loc:{label:'Location', all:'All locations', list:D.locations},
  gen:{label:'Gender', all:'All genders', list:D.genders}
};
function buildFilters(hostId, st, keys, onChange){
  const host = $(hostId);
  const html = keys.map(k => '<label class="pill-sel" data-k="' + k + '"><span>' + FDEF[k].label + '</span><select data-k="' + k + '">' +
    '<option value="all">' + FDEF[k].all + '</option>' +
    FDEF[k].list.map((c,i) => '<option value="' + i + '">' + esc(c.label || c.name) + ' (' + c.n + ')</option>').join('') +
    '</select></label>').join('') + '<button type="button" class="reset">Reset</button>';
  host.insertAdjacentHTML('afterbegin', html);
  const reset = host.querySelector('.reset');
  const sync = () => {
    host.querySelectorAll('label.pill-sel[data-k]').forEach(l => l.classList.toggle('on', st[l.dataset.k] !== 'all'));
    reset.classList.toggle('show', !keys.every(k => st[k] === 'all'));
  };
  host.querySelectorAll('select[data-k]').forEach(s => s.addEventListener('change', () => { st[s.dataset.k] = s.value; sync(); onChange(); }));
  reset.addEventListener('click', () => { keys.forEach(k => st[k] = 'all'); host.querySelectorAll('select[data-k]').forEach(s => s.value = 'all'); sync(); onChange(); });
}
const rowsFor = (st, keys) => ROWS.filter(r => keys.every(k => st[k] === 'all' || r[k] === +st[k]));
const isAll = (st, keys) => keys.every(k => st[k] === 'all');
function wireSeg(id, st, key, onChange){
  $(id).querySelectorAll('button').forEach(b => b.addEventListener('click', () => {
    st[key] = b.dataset.v;
    $(id).querySelectorAll('button').forEach(x => x.setAttribute('aria-pressed', x === b ? 'true' : 'false'));
    onChange();
  }));
}

const S = {
  hero:{dept:'all', ten:'all', mode:'fair'},
  c1:{dept:'all', ten:'all', loc:'all', gen:'all'},
  c2:{ten:'all', loc:'all', gen:'all', dim:'Fairness', hl:'all', size:'all'},
  c3:{dept:'all', loc:'all', gen:'all'},
  c4:{dept:'all', ten:'all'},
  c5:{dept:'all', ten:'all', out:'stay'}
};
const K = {hero:['dept','ten'], c1:['dept','ten','loc','gen'], c2:['ten','loc','gen'],
           c3:['dept','loc','gen'], c4:['dept','ten'], c5:['dept','ten']};

/* ---------- D. the balance ---------- */
const NS = 'http://www.w3.org/2000/svg';
const B = {W:96, H:34, DX:15, DY:9, BASE:373, LX:190, RX:450};
let bal = {l:0, r:0}, balTarget = null, firstDraw = true, tweenId = null;
function svg(n, a, p){ const e = document.createElementNS(NS, n); for(const k in a) e.setAttribute(k, a[k]); if(p) p.appendChild(e); return e; }
function drawStacks(lv, rv){
  const host = $('stacks'); host.innerHTML = '';
  [['left', B.LX, lv], ['right', B.RX, rv]].forEach(([side, cx, v]) => {
    const g = svg('g', {'class':'stack ' + side + ' m-' + S.hero.mode, 'data-side':side}, host);
    const x = cx - B.W/2;
    svg('rect', {x:x, y:B.BASE - 5*B.H, width:B.W, height:5*B.H, 'class':'ghost-col'}, g);
    const full = Math.floor(v + 1e-9), frac = v - full, count = Math.min(5, full + (frac > 0.02 ? 1 : 0));
    for(let i = 0; i < count; i++){
      const h = i < full ? B.H : B.H*frac, y = B.BASE - i*B.H - h;
      const b = svg('g', {'class':'blk' + (firstDraw ? ' blk-in' : ''), style:'--i:' + i}, g);
      svg('rect', {x:x, y:y, width:B.W, height:h, 'class':'f-front'}, b);
      svg('polygon', {points:`${x},${y} ${x+B.DX},${y-B.DY} ${x+B.W+B.DX},${y-B.DY} ${x+B.W},${y}`, 'class':'f-top'}, b);
      svg('polygon', {points:`${x+B.W},${y} ${x+B.W+B.DX},${y-B.DY} ${x+B.W+B.DX},${y+h-B.DY} ${x+B.W},${y+h}`, 'class':'f-side'}, b);
    }
    svg('rect', {x:x-8, y:B.BASE - 5*B.H - 16, width:B.W + B.DX + 16, height:5*B.H + 20, 'class':'hit'}, g);
  });
}
function balanceTarget(rows){
  if(!rows.length) return {empty:true, l:0, r:0, angle:0};
  const filtered = !isAll(S.hero, K.hero);
  if(S.hero.mode === 'fair'){
    const p = mean(rows.map(r => r.dim['Pride'])), f = mean(rows.map(r => r.dim['Fairness']));
    return {l:p, r:f, angle:-Math.max(-14, Math.min(14, (p - f)*10)),
      lName:'PRIDE', rName:'FAIRNESS', lVal:p.toFixed(2), rVal:f.toFixed(2),
      lSub: filtered ? 'company ' + COMPANY.dims['Pride'].toFixed(2) : 'highest of 6 dimensions',
      rSub: filtered ? 'company ' + COMPANY.dims['Fairness'].toFixed(2) : 'lowest of 6 dimensions'};
  }
  const uns = rows.reduce((a,r) => a + r.fairUnsettled, 0) / (rows.length * FAIR_ITEMS.length), set = 1 - uns;
  return {l:set*5, r:uns*5, angle:-Math.max(-14, Math.min(14, (set - uns)*14)),
    lName:'SETTLED', rName:'UNSETTLED', lVal:Math.round(set*100) + '%', rVal:Math.round(uns*100) + '%',
    lSub:'often or always', rSub: filtered ? 'company ' + Math.round(COMPANY.unsettled) + '%' : 'sometimes or worse'};
}
function updateBalance(){
  const rows = rowsFor(S.hero, K.hero), t = balanceTarget(rows); balTarget = t;
  $('heroN').textContent = plural(rows.length, 'person', 'people');
  $('flL').style.setProperty('--lc', S.hero.mode === 'fair' ? 'var(--sand)' : 'var(--teal-l)');
  $('flLName').textContent = t.empty ? '' : t.lName;  $('flRName').textContent = t.empty ? '' : t.rName;
  $('flLVal').textContent = t.empty ? 'n/a' : t.lVal; $('flRVal').textContent = t.empty ? 'n/a' : t.rVal;
  $('flLSub').textContent = t.empty ? '' : t.lSub;    $('flRSub').textContent = t.empty ? '' : t.rSub;
  $('modeCaption').textContent = t.empty ? 'No one matches these filters.'
    : S.hero.mode === 'fair' ? 'Average scores out of 5. One block = one point.' : 'Share of all fairness answers.';
  const beam = $('beam');
  if(firstDraw){
    bal = {l:t.l, r:t.r}; drawStacks(bal.l, bal.r); firstDraw = false;
    beam.style.transform = 'rotate(0deg)';
    setTimeout(() => { beam.style.transform = 'rotate(' + t.angle + 'deg)'; }, reduce ? 0 : 650);
    return;
  }
  beam.style.transform = 'rotate(' + t.angle + 'deg)';
  const from = {...bal}, t0 = performance.now(), dur = reduce ? 0 : 650;
  cancelAnimationFrame(tweenId);
  const step = now => {
    const p = dur ? Math.min(1, (now - t0)/dur) : 1, e = 1 - Math.pow(1 - p, 3);
    bal = {l:from.l + (t.l - from.l)*e, r:from.r + (t.r - from.r)*e};
    drawStacks(bal.l, bal.r);
    if(p < 1) tweenId = requestAnimationFrame(step);
  };
  tweenId = requestAnimationFrame(step);
}
function wireBalanceHover(){
  const el = $('balance'), tip = $('tip');
  const off = () => { el.classList.remove('hovering'); tip.classList.remove('on'); el.querySelectorAll('.stack').forEach(s => s.classList.remove('on')); };
  el.addEventListener('mousemove', ev => {
    const st = ev.target.closest ? ev.target.closest('.stack') : null, t = balTarget;
    if(!st || !t || t.empty) return off();
    el.querySelectorAll('.stack').forEach(s => s.classList.toggle('on', s === st));
    el.classList.add('hovering');
    const L = st.dataset.side === 'left';
    const body = S.hero.mode === 'fair'
      ? (L ? 'Pride: proud to work here, positive impact, going beyond.' : 'Fairness: rewards, growth, consistent decisions, workload, clarity.')
      : (L ? 'of fairness answers were often or always.' : 'of fairness answers were sometimes or worse.');
    tip.innerHTML = '<b>' + (L ? t.lVal : t.rVal) + '</b>' + body;
    tip.classList.add('on');
    tip.style.left = Math.min(window.innerWidth - 280, ev.clientX + 18) + 'px';
    tip.style.top = (ev.clientY + 18) + 'px';
  });
  el.addEventListener('mouseleave', off);
}

/* ---------- E. company overview tiles ---------- */
function drawOverview(){
  const ranked = DIM_NAMES.slice().sort((a,b) => COMPANY.dims[b] - COMPANY.dims[a]);
  const hi = ranked[0], lo = ranked[ranked.length - 1];
  $('ovTitle').innerHTML = 'Six dimensions measured. <b>' + lo + '</b> is the lowest, ' + hi.toLowerCase() + ' the highest.';
  const tile = (name, val, tag, color, frac, vc) => '<div class="dim" style="--dc:' + color + ';--vc:' + (vc || color) + '"><div class="d-name" title="' + name + '">' + name +
    '</div><div class="d-val num">' + val + '</div><div class="d-tag">' + tag + '</div><div class="d-bar"><i style="width:' + (100*frac).toFixed(1) + '%"></i></div></div>';
  $('dims').innerHTML = ranked.map(n => {        /* tiles run highest to lowest */
    const v = COMPANY.dims[n], tag = n === hi ? 'Highest' : n === lo ? 'Lowest' : 'Rank ' + (ranked.indexOf(n) + 1) + ' of 6';
    const color = n === hi ? 'var(--ochre)' : n === lo ? 'var(--rust)' : '#D3C6B1';
    return tile(n, v.toFixed(2), tag, color, (v - 1)/4, (n === hi || n === lo) ? color : 'var(--ink)');
  }).join('') + tile('Intend to stay', Math.round(COMPANY.stay) + '%', 'see themselves here in 3-5 yrs', 'var(--teal)', COMPANY.stay/100);
}

/* ---------- F. shared chart styling ---------- */
const FONT = 'Figtree, system-ui, sans-serif';
const CFG = {displayModeBar:false, responsive:true};
function layout(extra){
  return Object.assign({paper_bgcolor:'rgba(0,0,0,0)', plot_bgcolor:'rgba(0,0,0,0)',
    font:{family:FONT, color:C.ink, size:13}, margin:{l:64, r:24, t:16, b:56},
    hoverlabel:{bgcolor:C.card, bordercolor:C.line, font:{family:FONT, color:C.ink, size:13}},
    showlegend:false}, extra);
}
function axis(title, extra){
  return Object.assign({gridcolor:'rgba(36,27,18,.06)', zeroline:false, linecolor:C.line,
    tickfont:{color:C.muted, size:12}, title:{text:title, font:{color:C.muted, size:12.5}, standoff:12}}, extra || {});
}
function draw(cardId, traces, lay){
  const card = $(cardId), el = card.querySelector('.plot');
  card.classList.remove('is-empty');
  el.style.minHeight = (lay.height || 400) + 'px';      /* reserve space so the page never jumps */
  Plotly.react(el, traces, lay, CFG);
}
function empty(cardId, titleId, msg){
  const card = $(cardId), el0 = card.querySelector('.plot'), box = card.querySelector('.empty');
  box.style.minHeight = (el0.offsetHeight || parseInt(el0.style.minHeight, 10) || 420) + 'px';   /* same height as the chart it replaces */
  card.classList.add('is-empty');
  box.textContent = msg;
  $(titleId).textContent = 'Nothing to show for these filters';
  const el = card.querySelector('.plot'); if(el.data) Plotly.purge(el);
}
const setN = (id, n) => { $(id).textContent = plural(n, 'person', 'people'); };
const NOONE = 'No one matches these filters. Widen one, or press Reset.';
const DIVERGE = (lo, mid, hi) => { const p = Math.max(.05, Math.min(.95, (mid - lo)/(hi - lo)));
  return [[0,'#9A3F1C'],[p*.55,'#E0A07E'],[p,'#EFE6D6'],[p + (1-p)*.5,'#8DB8B2'],[1,'#1F5554']]; };
const DARK = v => v != null && (v < 62 || v >= 96);   /* bubble/bar colours dark enough to need light text */

/* ---------- G. Chart 1: retention across pride and fairness (soft targets) ---------- */
const FBINS = ['below 3.0','3.0 to 3.4','3.6 to 3.8','4.0 to 4.4','4.6 to 5.0'];
const PBINS = ['3.7 or less','4.0','4.3','4.7','5.0'];
const fBin = v => v < 2.99 ? 0 : v < 3.5 ? 1 : v < 3.95 ? 2 : v < 4.5 ? 3 : 4;
const pBin = v => v < 3.9 ? 0 : v < 4.1 ? 1 : v < 4.5 ? 2 : v < 4.8 ? 3 : 4;
const BUBBLE = '#5F7E7C';   /* one neutral colour: bubbles show HOW MANY, never a rate */
function drawC1(){
  const rows = rowsFor(S.c1, K.c1); setN('n1', rows.length);
  if(!rows.length) return empty('card1', 't1', NOONE);
  const base = COMPANY.stay, LO = 40, HI = 100, CS = DIVERGE(LO, base, HI);
  const cells = {};
  rows.forEach(r => { const k = fBin(r.dim['Fairness']) + '_' + pBin(r.dim['Pride']);
    (cells[k] = cells[k] || {n:0, s:0}); cells[k].n++; if(r.stays) cells[k].s++; });
  const keys = Object.keys(cells), maxN = Math.max(...keys.map(k => cells[k].n));
  const X = [], Y = [], SZ = [], TXT = [], HOV = [];
  keys.forEach(k => { const [fx, py] = k.split('_').map(Number), c = cells[k];
    X.push(FBINS[fx]); Y.push(PBINS[py]); SZ.push(16 + 56*Math.sqrt(c.n/maxN));
    TXT.push(c.n >= 2 ? String(c.n) : '');
    HOV.push('Fairness ' + FBINS[fx] + ', pride ' + PBINS[py] + '<br><b>' + plural(c.n, 'person', 'people') + '</b>' +
      (c.n >= 8 ? '<br>' + Math.round(100*c.s/c.n) + '% intend to stay' : '')); });
  /* retention is read from the margins only, where each bar pools enough people to be reliable */
  const col = FBINS.map((_, i) => stayPct(rows.filter(r => fBin(r.dim['Fairness']) === i)));
  const row = PBINS.map((_, i) => stayPct(rows.filter(r => pBin(r.dim['Pride']) === i)));
  const colN = FBINS.map((_, i) => rows.filter(r => fBin(r.dim['Fairness']) === i).length);
  const rowN = PBINS.map((_, i) => rows.filter(r => pBin(r.dim['Pride']) === i).length);
  const quad = (pf, pp) => { const g = rows.filter(r => (r.dim['Fairness'] >= 3.95) === pf && (r.dim['Pride'] >= 4.2) === pp); return {n:g.length, stay:stayPct(g)}; };
  const q = {pnf:quad(false,true), pf:quad(true,true), npf:quad(true,false), npnf:quad(false,false)};
  const qbox = (name, g, color) => {
    const d = g.n ? g.stay - base : null;
    return '<div class="quad" style="--qc:' + color + '"><div class="q-name">' + name + '</div><div class="q-val num ' + (d == null ? '' : d >= 0 ? 'up' : 'down') + '">' +
      (g.n ? Math.round(g.stay) + '% stay' : 'n/a') + '</div><div class="q-sub">' + plural(g.n, 'person', 'people') + (d == null ? '' : ', ' + pts(d) + ' vs company') + '</div></div>'; };
  $('quads1').innerHTML = qbox('Soft targets: proud, not fairly treated', q.pnf, C.rust) + qbox('Proud and fairly treated', q.pf, '#D3C6B1') +
                          qbox('Fairly treated, not proud', q.npf, '#D3C6B1') + qbox('Neither', q.npnf, '#D3C6B1');
  const lowF = rows.filter(r => r.dim['Fairness'] < 3.95), hiF = rows.filter(r => r.dim['Fairness'] >= 3.95);
  $('t1').innerHTML = (lowF.length && hiF.length)
    ? 'Fairness splits retention: <i>' + Math.round(stayPct(hiF)) + '%</i> stay when it scores 4+, <b>' + Math.round(stayPct(lowF)) + '%</b> when it does not'
    : 'Everyone here rates fairness ' + (hiF.length ? '4 or above' : 'below 4') + ': ' + Math.round(stayPct(rows)) + '% intend to stay';
  const marg = {colorscale:CS, cmin:LO, cmax:HI};
  draw('card1', [
    {type:'scatter', mode:'markers', x:X, y:Y, hovertext:HOV, hoverinfo:'text',
      marker:{size:SZ, sizemode:'diameter', color:BUBBLE, opacity:.9, line:{width:1, color:'#FFFCF6'}}},
    /* counts as their own text trace: markers+text silently drops labels on redraw, this never does */
    {type:'scatter', mode:'text', x:X.filter((_,k) => TXT[k]), y:Y.filter((_,k) => TXT[k]),
      text:TXT.filter(t => t), textfont:{size:12, color:'#FFFCF6', family:FONT}, hoverinfo:'skip'},
    {type:'bar', x:FBINS, y:col.map(v => v == null ? 0 : v), xaxis:'x', yaxis:'y2', width:.52,
      marker:Object.assign({color:col.map(v => v == null ? base : v)}, marg),
      text:col.map(v => v == null ? '' : Math.round(v) + '%'), textposition:'inside', insidetextanchor:'end', textangle:0,
      textfont:{size:11.5, color:col.map(v => DARK(v) ? C.cream : C.ink)},
      hovertext:col.map((v,i) => 'Fairness ' + FBINS[i] + '<br>' + plural(colN[i], 'person', 'people') + (v == null ? '' : ', ' + Math.round(v) + '% intend to stay')), hoverinfo:'text'},
    {type:'bar', orientation:'h', x:row.map(v => v == null ? 0 : v), y:PBINS, xaxis:'x3', yaxis:'y', width:.52,
      marker:Object.assign({color:row.map(v => v == null ? base : v)}, marg),
      text:row.map(v => v == null ? '' : Math.round(v) + '%'), textposition:'inside', insidetextanchor:'end', textangle:0,
      textfont:{size:11.5, color:row.map(v => DARK(v) ? C.cream : C.ink)},
      hovertext:row.map((v,i) => 'Pride ' + PBINS[i] + '<br>' + plural(rowN[i], 'person', 'people') + (v == null ? '' : ', ' + Math.round(v) + '% intend to stay')), hoverinfo:'text'}],
    layout({height:560, margin:{l:112, r:34, t:12, b:60},
      xaxis:axis('Fairness score', {type:'category', categoryorder:'array', categoryarray:FBINS, range:[-.6, 4.6], domain:[0, .84], showgrid:false}),
      yaxis:axis('Pride score', {type:'category', categoryorder:'array', categoryarray:PBINS, range:[-.6, 4.6], domain:[0, .78], showgrid:false}),
      yaxis2:{domain:[.87, 1], anchor:'x', range:[0, 104], showticklabels:false, showgrid:false, zeroline:false},
      xaxis3:{domain:[.885, 1], anchor:'y', range:[0, 104], showticklabels:false, showgrid:false, zeroline:false},
      shapes:[
        {type:'rect', xref:'x', yref:'y', x0:-.6, x1:2.5, y0:1.5, y1:4.6, fillcolor:'rgba(200,87,46,.05)', line:{width:0}, layer:'below'},
        {type:'line', xref:'paper', yref:'y2', x0:0, x1:.84, y0:base, y1:base, line:{color:'rgba(36,27,18,.3)', width:1, dash:'dot'}, layer:'below'},
        {type:'line', xref:'x3', yref:'paper', x0:base, x1:base, y0:0, y1:.78, line:{color:'rgba(36,27,18,.3)', width:1, dash:'dot'}, layer:'below'}],
      annotations:[
        {xref:'paper', yref:'paper', x:-.015, y:.935, xanchor:'right', showarrow:false, align:'right', text:'Intend to stay,<br>by fairness', font:{size:11, color:C.muted}},
        {xref:'paper', yref:'paper', x:.885, y:.79, xanchor:'left', yanchor:'bottom', showarrow:false, text:'Intend to stay,<br>by pride', font:{size:11, color:C.muted}},
        {xref:'paper', yref:'y2', x:.84, y:base, xanchor:'left', xshift:5, showarrow:false, text:'company<br>' + Math.round(base) + '%', font:{size:10.5, color:C.muted}},
        {xref:'x', yref:'y', x:-.5, y:4.5, xanchor:'left', yanchor:'top', showarrow:false, text:'<b>SOFT TARGETS</b>  proud, not fairly treated', font:{size:11, color:C.rust}}]}));
}


/* ---------- H. Chart 2: departments against the company ---------- */
function drawC2(){
  const st = S.c2, rows = rowsFor(st, K.c2), dim = st.dim; setN('n2', rows.length);
  if(!rows.length) return empty('card2', 't2', NOONE);
  const comp = mean(rows.map(r => r.dim[dim])), hl = st.hl === 'all' ? -1 : +st.hl, MIN = 5;
  const HIDDEN = D.depts.map((d,i) => /disclose/i.test(d.name) ? i : -1).filter(i => i >= 0);   /* not a department */
  const nHidden = rows.filter(r => HIDDEN.includes(r.dept)).length;
  $('note2').textContent = nHidden ? 'Not shown: ' + plural(nHidden, 'person', 'people') + ' chose not to disclose a department. Their answers still count in the company average.' : '';
  let g = D.depts.map((d,i) => Object.assign({i:i, name:d.name}, summarise(rows.filter(r => r.dept === i).map(r => r.dim[dim]))))
                 .filter(x => x.n > 0 && !HIDDEN.includes(x.i) && (st.size === 'all' || x.n >= MIN));
  if(!g.length) return empty('card2', 't2', 'No department has 5 or more people under these filters. Switch to All teams.');
  g.sort((a,b) => b.mean - a.mean || a.n - b.n);
  const below = g.filter(x => x.n >= MIN && x.half != null && x.mean + x.half < comp);
  const above = g.filter(x => x.n >= MIN && x.half != null && x.mean - x.half > comp);
  const col = x => below.includes(x) ? C.rust : above.includes(x) ? C.teal : C.stone;
  const lab = g.map(x => (x.i === hl ? '\u25B6 ' : '') + x.name + '  (' + x.n + ')');
  const lows = g.map(x => x.half == null ? x.mean : Math.max(1, x.mean - x.half));
  const xmin = Math.max(1, Math.min(...lows) - .12);
  const worst = below.slice().sort((a,b) => a.mean - b.mean)[0];
  $('t2').innerHTML = worst
    ? '<b>' + esc(worst.name) + '</b> sits ' + (comp - worst.mean).toFixed(2) + ' below the company on ' + dim.toLowerCase()
    : 'No department sits clearly below the company on ' + dim.toLowerCase();
  const ann = [{x:comp, y:1, xref:'x', yref:'paper', yanchor:'bottom', showarrow:false, text:'<b>company ' + comp.toFixed(2) + '</b>', font:{size:12, color:C.ink}},
               {xref:'paper', yref:'paper', x:1, xshift:14, y:1, xanchor:'left', yanchor:'bottom', showarrow:false, text:'avg', font:{size:11, color:C.muted}}]
    .concat(g.map((x,k) => ({xref:'paper', yref:'y', x:1, xshift:14, y:lab[k], xanchor:'left', showarrow:false,
      text:(below.includes(x) || above.includes(x) ? '<b>' : '') + x.mean.toFixed(2) + (below.includes(x) || above.includes(x) ? '</b>' : ''),
      font:{size:12, color:x.n < MIN ? C.faint : col(x)}})));
  if(worst) ann.push({x:worst.mean, y:lab[g.indexOf(worst)], xref:'x', yref:'y', ax:-120, ay:0, showarrow:true, arrowhead:0,
    arrowcolor:C.rust, arrowwidth:1.3, text:(worst.mean - comp).toFixed(2) + ' vs company', font:{size:12, color:C.rust},
    bgcolor:'#FFFCF6', borderpad:3});
  draw('card2', [{type:'scatter', mode:'markers', x:g.map(x => x.mean), y:lab,
      error_x:{type:'data', array:g.map(x => x.half == null ? 0 : x.half), visible:true, color:'rgba(118,102,79,.3)', thickness:2, width:0},
      marker:{size:g.map(x => Math.max(8, Math.min(19, 6 + 1.6*Math.sqrt(x.n)))), color:g.map(col), opacity:g.map(x => x.n < MIN ? .38 : 1),
        line:{width:g.map(x => x.i === hl ? 3 : 1), color:g.map(x => x.i === hl ? C.ink : '#FFFCF6')}},
      hovertext:g.map(x => '<b>' + esc(x.name) + '</b><br>' + dim + ' ' + x.mean.toFixed(2) + ' (' + (x.mean - comp >= 0 ? '+' : '') + (x.mean - comp).toFixed(2) + ' vs company)' +
        (x.half == null ? '' : '<br>95% range ' + Math.max(1, x.mean - x.half).toFixed(2) + ' to ' + Math.min(5, x.mean + x.half).toFixed(2)) + '<br>' + plural(x.n, 'person', 'people')),
      hoverinfo:'text'}],
    layout({height:Math.max(380, 96 + 26*(st.size === 'all' ? D.depts.length - 1 : D.depts.filter(d => d.n >= MIN).length)), margin:{l:20, r:64, t:40, b:52},
      xaxis:axis(dim + ' score', {range:[xmin, 5.05]}),
      yaxis:axis('', {type:'category', automargin:true, showgrid:false, tickfont:{size:12.5, color:C.ink}}),
      shapes:[{type:'rect', xref:'x', yref:'paper', x0:xmin, x1:comp, y0:0, y1:1, fillcolor:'rgba(200,87,46,.03)', line:{width:0}, layer:'below'},
              {type:'line', xref:'x', yref:'paper', x0:comp, x1:comp, y0:0, y1:1, line:{color:C.ink, width:1.4, dash:'dash'}}],
      annotations:ann}));
}

/* ---------- I. Chart 3: pride and fairness across years of service ---------- */
function drawC3(){
  const rows = rowsFor(S.c3, K.c3); setN('n3', rows.length);
  if(!rows.length) return empty('card3', 't3', NOONE);
  const bands = D.tenures.map((t,i) => { const b = rows.filter(r => r.ten === i);
    return {label:t.label, name:t.name, n:b.length, f:summarise(b.map(r => r.dim['Fairness'])), p:summarise(b.map(r => r.dim['Pride']))}; });
  const xl = bands.map(b => b.label + '<br><span style="font-size:11px;color:#6B6257">n = ' + b.n + '</span>');
  const traces = [];
  [['Pride','p',C.ochre,'rgba(168,131,63,.15)','top center'], ['Fairness','f',C.rust,'rgba(200,87,46,.12)','bottom center']].forEach(([name,k,c,fill,tp]) => {
    const m = bands.map(b => b[k].mean), h = bands.map(b => b[k].half);
    traces.push({x:xl, y:m.map((v,i) => v == null ? null : Math.min(5, v + (h[i] || 0))), mode:'lines', line:{width:0}, hoverinfo:'skip'});
    traces.push({x:xl, y:m.map((v,i) => v == null ? null : Math.max(1, v - (h[i] || 0))), mode:'lines', line:{width:0}, fill:'tonexty', fillcolor:fill, hoverinfo:'skip'});
    traces.push({x:xl, y:m, mode:'lines+markers+text', line:{color:c, width:4, shape:'spline', smoothing:.5},
      marker:{size:11, color:c, line:{width:2.5, color:'#FFFCF6'}}, text:m.map(v => v == null ? '' : v.toFixed(2)),
      textposition:tp, textfont:{size:12.5, color:c, family:FONT},
      hovertext:bands.map(b => '<b>' + name + ', ' + b.label + '</b><br>' + (b[k].mean == null ? 'No one' : b[k].mean.toFixed(2)) + '<br>' + plural(b.n, 'person', 'people')), hoverinfo:'text'});
  });
  const valid = bands.filter(b => b.f.mean != null);
  const vals = bands.flatMap(b => [b.f.mean, b.p.mean, b.f.half ? b.f.mean - b.f.half : null]).filter(v => v != null);
  const ymin = Math.max(1, Math.min(...vals) - .2);
  const ann = [{xref:'paper', yref:'y', x:0, y:valid.length ? valid[0].p.mean : 5, xanchor:'left', yanchor:'bottom', yshift:16, showarrow:false, text:'<b>Pride</b>', font:{size:12.5, color:C.ochre}},
    {xref:'paper', yref:'y', x:0, y:valid.length ? valid[0].f.mean : 4, xanchor:'left', yanchor:'top', yshift:-16, showarrow:false, text:'<b>Fairness</b>', font:{size:12.5, color:C.rust}}];
  if(isAll(S.c3, K.c3)){
    D.full.tenure_steps.filter(s => s.significant).forEach(s => {
      const i = D.tenures.findIndex(t => t.name === s.to); if(i < 0 || bands[i].f.mean == null) return;
      ann.push({x:xl[i], y:bands[i].f.mean, ax:s.change > 0 ? -70 : 60, ay:s.change > 0 ? 62 : 58, showarrow:true, arrowhead:0,
        arrowwidth:1.2, arrowcolor:C.muted, bgcolor:'#FFFCF6', bordercolor:C.line, borderpad:4,
        text:'<b>' + (s.change > 0 ? '+' : '') + s.change.toFixed(2) + '</b> real change (p = ' + s.p + ')', font:{size:11.5, color:C.ink}});
    });
  }
  const peak = valid.reduce((a,b) => b.f.mean > a.f.mean ? b : a, valid[0]), last = valid[valid.length - 1];
  const low = valid.reduce((a,b) => b.f.mean < a.f.mean ? b : a, valid[0]);
  $('t3').innerHTML = (valid.length > 1 && peak !== last && peak.f.mean - last.f.mean >= .05)
    ? 'Fairness peaks at <i>' + peak.label + '</i> (' + peak.f.mean.toFixed(2) + '), then drops <b>' + (peak.f.mean - last.f.mean).toFixed(2) + '</b> for ' + last.label
    : 'Fairness is lowest at <b>' + low.label + '</b> (' + low.f.mean.toFixed(2) + ')';
  draw('card3', traces, layout({height:480, margin:{l:56, r:24, t:20, b:72},
    xaxis:axis('', {type:'category', showgrid:false}),
    yaxis:axis('Average score (out of 5)', {range:[ymin, 5.08]}),
    annotations:ann}));
}

/* ---------- J. Chart 4: what to work on to raise fairness ---------- */
function drawC4(){
  const st = S.c4, rows = rowsFor(st, K.c4); setN('n4', rows.length);
  if(!rows.length) return empty('card4', 't4', NOONE);
  const A = rows.filter(r => r.dim['Fairness'] < 3.95), Bq = rows.filter(r => r.dim['Fairness'] >= 3.95);
  if(!A.length) return empty('card4', 't4', 'No one under these filters rates fairness below 4.');
  const share = (g, j) => g.length ? 100*g.filter(r => r.th[j]).length/g.length : null;
  const T = D.themes.map((t,j) => ({name:t.name, need:t.need, a:share(A, j), b:share(Bq, j)}))
                    .map(t => Object.assign(t, {d:t.b == null ? null : t.a - t.b}));
  T.sort((x,y) => y.a - x.a);                                   /* most-asked-for first */
  const zOf = t => { if(t.b == null) return 0;                  /* two-proportion test: is the gap bigger than chance? */
    const p1 = t.a/100, p2 = t.b/100, n1 = A.length, n2 = Bq.length, p = (p1*n1 + p2*n2)/(n1 + n2), se = Math.sqrt(p*(1-p)*(1/n1 + 1/n2));
    return se ? (p1 - p2)/se : 0; };
  const LEVER = t => t.d != null && zOf(t) >= 2.8;              /* clearly more than fairly treated colleagues; 2.8 allows for comparing 10 themes at once */
  const y = T.map(t => t.name + (t.need === 'Budget' ? '  \u00B7 budget' : ''));
  const levers = T.filter(LEVER);
  $('t4').innerHTML = levers.length
    ? 'To raise fairness, start with <b>' + levers.slice(0, 2).map(t => t.name.toLowerCase()).join('</b>, then <b>') + '</b>'
    : 'People who feel unfairly treated ask for much the same as everyone else';
  draw('card4', [{type:'bar', orientation:'h', x:T.map(t => t.a), y:y, width:.62,
      marker:{color:T.map(t => LEVER(t) ? C.rust : '#DCCFB9')},
      text:T.map(t => Math.round(t.a) + '%' + (LEVER(t) ? '   ' + pts(t.d) + ' vs colleagues' : '')),
      textposition:'outside', cliponaxis:false, textfont:{size:12, color:T.map(t => LEVER(t) ? C.rust : C.muted)},
      hovertext:T.map(t => '<b>' + t.name + '</b><br>' + Math.round(t.a) + '% of people who feel unfairly treated' +
        (t.b == null ? '' : '<br>' + Math.round(t.b) + '% of colleagues who feel fairly treated')), hoverinfo:'text'}],
    layout({height:470, margin:{l:20, r:30, t:14, b:52},
      xaxis:axis('% of people who rate fairness below 4 who mention it', {range:[0, Math.max(10, ...T.map(t => t.a)) * 1.42], ticksuffix:'%'}),
      yaxis:axis('', {type:'category', autorange:'reversed', automargin:true, showgrid:false, tickfont:{size:12.5, color:C.ink}})}));
}


/* ---------- K. Chart 5: which questions to settle first ---------- */
function drawC5(){
  const st = S.c5, rows = rowsFor(st, K.c5); setN('n5', rows.length);
  if(!rows.length) return empty('card5', 't5', NOONE);
  if(rows.length < 3) return empty('card5', 't5', 'Correlations need at least 3 people. Widen the filters.');
  const outIx = st.out === 'stay' ? D.outcomes.stay : D.outcomes.recommend;
  const outName = st.out === 'stay' ? 'staying' : 'recommending';
  const yv = rows.map(r => r.s[outIx]), P = [];
  D.items.forEach((it,i) => { if(it.dim === 'Outcome') return;
    const r = pearson(rows.map(x => x.s[i]), yv); if(r == null) return;
    P.push({i:i, it:it, x:100*rows.filter(x => x.s[i] < 4).length/rows.length, r:r}); });
  if(P.length < 3) return empty('card5', 't5', 'Everyone here answered the outcome question the same way, so there is nothing to correlate.');
  const med = a => { const s = a.slice().sort((p,q) => p - q), m = s.length >> 1; return s.length % 2 ? s[m] : (s[m-1] + s[m])/2; };
  const mx = med(P.map(p => p.x)), my = med(P.map(p => p.r));
  P.forEach(p => p.score = (p.x/100) * p.r);
  const top = P.filter(p => p.x >= mx && p.r >= my).sort((a,b) => b.score - a.score);
  const show = new Set(top.slice(0, 3).map(p => p.i).concat(P.filter(p => p.it.dim === 'Fairness').map(p => p.i)));
  const POS = ['top center','bottom center','middle right','top left','bottom right','middle left'];
  let k = 0;
  const traces = DIM_ALPHA.map(dn => {
    const Q = P.filter(p => p.it.dim === dn), F = dn === 'Fairness';
    return {type:'scatter', mode:'markers+text', name:dn, x:Q.map(p => p.x), y:Q.map(p => p.r),
      text:Q.map(p => show.has(p.i) ? p.it.short : ''), textposition:Q.map(() => POS[k++ % POS.length]),
      textfont:{size:12, color:F ? C.rust : C.ink, family:FONT},
      marker:{size:Q.map(p => show.has(p.i) ? (F ? 19 : 14) : 10), color:DIM_COLORS[dn], opacity:Q.map(p => show.has(p.i) ? 1 : .55),
        symbol:F ? 'diamond' : 'circle', line:{width:1.5, color:'#FFFCF6'}},
      hovertext:Q.map(p => '<b>' + esc(p.it.short) + '</b> (' + dn + ')<br>' + wrap(esc(p.it.question), 46) +
        '<br>' + Math.round(p.x) + '% sometimes or worse, r = ' + p.r.toFixed(2)), hoverinfo:'text'};
  });
  const xs = P.map(p => p.x), ys = P.map(p => p.r);
  const xr = [Math.max(0, Math.min(...xs) - 3), Math.max(...xs) + 5], yr = [Math.min(...ys) - .05, Math.max(...ys) + .07];
  $('t5').innerHTML = top.length
    ? 'Settle first: <b>' + top.slice(0, 3).map(p => esc(p.it.short).toLowerCase()).join(', ') + '</b>'
    : 'No question is both widespread and strongly tied to ' + outName + ' here';
  draw('card5', traces, layout({height:600, showlegend:true, margin:{l:64, r:24, t:40, b:60},
    legend:{orientation:'h', x:0, y:1.075, font:{size:12}},
    xaxis:axis('How widespread: % answering sometimes or worse', {range:xr, ticksuffix:'%'}),
    yaxis:axis('Tied to ' + outName + ' (correlation r)', {range:yr}),
    shapes:[{type:'rect', xref:'x', yref:'y', x0:mx, x1:xr[1], y0:my, y1:yr[1], fillcolor:'rgba(200,87,46,.06)', line:{width:0}, layer:'below'},
            {type:'line', xref:'x', yref:'paper', x0:mx, x1:mx, y0:0, y1:1, line:{color:'rgba(23,19,15,.28)', width:1.2, dash:'dot'}},
            {type:'line', xref:'paper', yref:'y', x0:0, x1:1, y0:my, y1:my, line:{color:'rgba(23,19,15,.28)', width:1.2, dash:'dot'}}],
    annotations:[{x:1, y:1, xref:'paper', yref:'paper', xanchor:'right', yanchor:'top', showarrow:false, text:'<b>SETTLE FIRST</b>', font:{size:12, color:C.rust}},
                 {x:0, y:1, xref:'paper', yref:'paper', xanchor:'left', yanchor:'top', showarrow:false, text:'<b>PROTECT</b>', font:{size:11.5, color:C.muted}},
                 {x:1, y:0, xref:'paper', yref:'paper', xanchor:'right', yanchor:'bottom', showarrow:false, text:'<b>WIDESPREAD, WEAKER LINK</b>', font:{size:11.5, color:C.muted}},
                 {x:0, y:0, xref:'paper', yref:'paper', xanchor:'left', yanchor:'bottom', showarrow:false, text:'<b>LOWER PRIORITY</b>', font:{size:11.5, color:C.muted}},
                 {x:mx, y:yr[0], xref:'x', yref:'y', yanchor:'bottom', xanchor:'left', xshift:4, showarrow:false, text:'median', font:{size:10.5, color:C.muted}}]}));
}

/* ---------- L. start-up: build every chart's own filters, then draw ---------- */
function safe(fn){ try { fn(); } catch(err){ console.error((fn.name || 'draw') + ' failed:', err); } }
const pending = new Map();
function schedule(fn){                       /* one redraw per frame per chart, however fast the clicks */
  if(pending.has(fn)) return;
  pending.set(fn, requestAnimationFrame(() => { pending.delete(fn); safe(fn); }));
}
buildFilters('fHero', S.hero, K.hero, () => schedule(updateBalance));
buildFilters('f1', S.c1, K.c1, () => schedule(drawC1));
buildFilters('f2', S.c2, K.c2, () => schedule(drawC2));
buildFilters('f3', S.c3, K.c3, () => schedule(drawC3));
buildFilters('f4', S.c4, K.c4, () => schedule(drawC4));
buildFilters('f5', S.c5, K.c5, () => schedule(drawC5));
document.querySelectorAll('.mode-btn').forEach(b => b.addEventListener('click', () => {
  S.hero.mode = b.dataset.mode;
  document.querySelectorAll('.mode-btn').forEach(x => x.setAttribute('aria-pressed', x === b ? 'true' : 'false'));
  safe(updateBalance);
}));
$('c2dim').innerHTML = DIM_ALPHA.map(n => '<option' + (n === 'Fairness' ? ' selected' : '') + '>' + n + '</option>').join('');
$('c2dim').addEventListener('change', e => { S.c2.dim = e.target.value; e.target.closest('.pill-sel').classList.toggle('on', S.c2.dim !== 'Fairness'); safe(drawC2); });
$('c2hl').innerHTML = '<option value="all">None</option>' + D.depts.map((d,i) => '<option value="' + i + '">' + esc(d.name) + '</option>').join('');
$('c2hl').addEventListener('change', e => { S.c2.hl = e.target.value; e.target.closest('.pill-sel').classList.toggle('on', S.c2.hl !== 'all'); safe(drawC2); });
wireSeg('c2size', S.c2, 'size', () => schedule(drawC2));
wireSeg('c5seg', S.c5, 'out', () => schedule(drawC5));
wireBalanceHover();
safe(drawOverview); safe(updateBalance);
[drawC1, drawC2, drawC3, drawC4, drawC5].forEach(safe);
window.__ledger = {S:S, K:K, rowsFor:rowsFor};
})();
"""

print("Cell 8 done: browser logic ready.")

Cell 8 done: browser logic ready.


## Cell 9: Page structure (HTML)
Placeholders like `__PAYLOAD__` are filled in Cell 10 with `.replace()`, so CSS and JavaScript braces never clash with Python formatting.

In [10]:
PAGE_HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>The Fairness Ledger: Safewater Lines</title>
__FONTS__
__PLOTLY__
<style>__CSS__</style>
</head>
<body>

<!-- ================= HERO ================= -->
<header class="hero" id="hero">
  <div class="hero-top">
    <div><span class="brand">The Fairness Ledger &middot; Safewater Lines</span>
      <h1>Proud of the company. <em>Owed</em> by it.</h1></div>
    <div class="hero-filters controls" id="fHero" style="margin:0"><span class="c-n" id="heroN" style="background:rgba(246,236,218,.08);color:var(--sand)"></span></div>
  </div>

  <div class="stage">
    <div class="watermark" aria-hidden="true">LEDGER</div>
    <div class="flank l" id="flL"><span class="fl-name" id="flLName"></span><span class="fl-val" id="flLVal"></span><span class="fl-sub" id="flLSub"></span></div>
    <svg class="balance" id="balance" viewBox="70 120 500 340" preserveAspectRatio="xMidYMid meet" role="img"
         aria-label="A balance scale. In FAIR mode it weighs pride against fairness; in DEBT mode, settled against unsettled fairness answers.">
      <g class="ground"><line x1="80" y1="446" x2="560" y2="446"/><polygon points="320,387 286,446 354,446"/></g>
      <g class="beam" id="beam">
        <rect class="beam-bar" x="90" y="373" width="460" height="14" rx="4"/>
        <g id="stacks"></g>
      </g>
      <circle class="pivot" cx="320" cy="380" r="8"/>
    </svg>
    <div class="flank r"><span class="fl-name" id="flRName"></span><span class="fl-val" id="flRVal"></span><span class="fl-sub" id="flRSub"></span></div>
  </div>

  <div class="hero-foot">
    <p class="mode-caption" id="modeCaption"></p>
    <div class="mode-toggle" role="group" aria-label="What the scale weighs">
      <button class="mode-btn" type="button" data-mode="fair" aria-pressed="true">FAIR</button>
      <button class="mode-btn" type="button" data-mode="debt" aria-pressed="false">DEBT</button>
    </div>
    <a class="explore" href="#definitions">Explore the data</a>
  </div>
</header>
<div class="tip" id="tip" role="tooltip"></div>

<main class="wrap">

  <!-- ================= DEFINITIONS: read this first ================= -->
  <section class="section" id="definitions">
    <p class="s-label">Read this first</p>
    <h2 class="ov-title">What the terms on this dashboard mean</h2>
    <div class="defs">
      <div class="def"><h3>Score (1 to 5)</h3><p>Every answer becomes a score: always or almost always = 5, often or usually = 4, sometimes = 3, rarely = 2, almost never = 1.</p></div>
      <div class="def key"><h3>Fairness</h3><p>The average of five questions: rewards are fair, equal chance to grow, decisions without favouritism, workload shared fairly, and understanding how decisions are made.</p></div>
      <div class="def"><h3>Pride</h3><p>The average of three questions: proud to say I work here, the company has a positive impact on customers, and willing to go beyond when needed.</p></div>
      <div class="def"><h3>Other dimensions</h3><p>Camaraderie (teamwork and belonging), Credibility (trust in leaders), Overall experience (growth, resources, satisfaction, meaningful work) and Respect (support and appreciation). Each is an average of 4 or 5 questions.</p></div>
      <div class="def key"><h3>Debt and settled</h3><p>A fairness answer of often or always is <b>settled</b>. An answer of sometimes or worse is <b>debt</b>: fairness the company still owes its people.</p></div>
      <div class="def"><h3>Intend to stay</h3><p>Answered often or always to &ldquo;I see myself continuing to work here over the next 3 to 5 years&rdquo;.</p></div>
      <div class="def"><h3>Proud and fairly treated</h3><p><b>Proud</b> means a pride score of 4.3 or higher. <b>Fairly treated</b> means a fairness score of 4.0 or higher.</p></div>
      <div class="def key"><h3>Soft targets</h3><p>Proud, but not fairly treated. Loyal today, yet the easiest people to lose to a better offer if fairness does not improve.</p></div>
      <div class="def"><h3>95% range</h3><p>The band where the true average most likely sits. If two ranges overlap, the difference between them could be chance.</p></div>
    </div>
  </section>

  <!-- ================= COMPANY OVERVIEW ================= -->
  <section class="section" id="overview">
    <p class="s-label">Company-wide &middot; __N__ people</p>
    <h2 class="ov-title" id="ovTitle"></h2>
    <div class="dims" id="dims"></div>
  </section>

  <!-- ================= CHART 1 ================= -->
  <section class="section"><div class="card" id="card1">
    <div class="card-top"><div><p class="s-label">01 &middot; Who are the soft targets</p><h2 class="c-title" id="t1"></h2>
      <p class="c-sub">Bubble size = how many people sit at that pair of scores. The bars outside the grid show who intends to stay.</p></div><span class="c-n" id="n1"></span></div>
    <div class="controls" id="f1"></div>
    <div class="quads" id="quads1"></div>
    <div class="plot"></div><p class="empty"></p>
  </div></section>

  <!-- ================= CHART 2 ================= -->
  <section class="section"><div class="card" id="card2">
    <div class="card-top"><div><p class="s-label">02 &middot; Where it concentrates</p><h2 class="c-title" id="t2"></h2>
      <p class="c-sub">Dot = department average, line = 95% range. Faded dots have fewer than 5 people.</p></div><span class="c-n" id="n2"></span></div>
    <div class="controls" id="f2">
      <span class="ctl-sep"></span>
      <label class="pill-sel"><span>Measure</span><select id="c2dim"></select></label>
      <label class="pill-sel"><span>Highlight</span><select id="c2hl"></select></label>
      <div class="seg" id="c2size" role="group" aria-label="Team size">
        <button type="button" data-v="all" aria-pressed="true">All teams</button>
        <button type="button" data-v="5" aria-pressed="false">5+ people</button>
      </div>
    </div>
    <div class="plot"></div><p class="empty"></p>
    <p class="note" id="note2"></p>
  </div></section>

  <!-- ================= CHART 3 ================= -->
  <section class="section"><div class="card" id="card3">
    <div class="card-top"><div><p class="s-label">03 &middot; How long it has been building</p><h2 class="c-title" id="t3"></h2>
      <p class="c-sub">Shaded bands are 95% ranges. Labelled steps are tested changes, shown when no filter is set.</p></div><span class="c-n" id="n3"></span></div>
    <div class="controls" id="f3"></div>
    <div class="plot"></div><p class="empty"></p>
  </div></section>

  <!-- ================= CHART 4 ================= -->
  <section class="section"><div class="card" id="card4">
    <div class="card-top"><div><p class="s-label">04 &middot; What would raise fairness</p><h2 class="c-title" id="t4"></h2>
      <p class="c-sub">What people who rate fairness below 4 asked for in their comments. Orange: asked for clearly more than colleagues who feel fairly treated.</p></div><span class="c-n" id="n4"></span></div>
    <div class="controls" id="f4"></div>
    <div class="plot"></div><p class="empty"></p>
  </div></section>

  <!-- ================= CHART 5 ================= -->
  <section class="section"><div class="card" id="card5">
    <div class="card-top"><div><p class="s-label">05 &middot; What to settle first</p><h2 class="c-title" id="t5"></h2>
      <p class="c-sub">All 28 questions. Diamonds are fairness. Correlation shows what moves together, not proof of cause.</p></div><span class="c-n" id="n5"></span></div>
    <div class="controls" id="f5">
      <span class="ctl-sep"></span>
      <div class="seg" id="c5seg" role="group" aria-label="Outcome">
        <button type="button" data-v="stay" aria-pressed="true">Intent to stay</button>
        <button type="button" data-v="recommend" aria-pressed="false">Would recommend</button>
      </div>
    </div>
    <div class="plot"></div><p class="empty"></p>
  </div></section>

</main>
<footer><div>Built from __N__ responses to the Safewater Lines employee survey. No names or comment text are included. TISS live industry project: From Employee Voices to Organisational Impact.</div></footer>

<script>window.LEDGER = __PAYLOAD__;</script>
<script>__JS__</script>
</body>
</html>"""

print("Cell 9 done: page structure ready.")

Cell 9 done: page structure ready.


## Cell 10: Build the dashboard and open it
Embeds Plotly's JavaScript so the page works offline, fills every placeholder, checks none are left, writes the HTML file and opens it in your browser. If plotly is not installed it falls back to loading Plotly from the internet and tells you.

In [ ]:
try:
    import plotly
    plotly_js = (Path(plotly.__file__).parent / "package_data" / "plotly.min.js").read_text(encoding="utf-8")
    PLOTLY_TAG = "<script>" + plotly_js.replace("</script", "<\\/script") + "</script>"
    plotly_note = f"Plotly {plotly.__version__} embedded: the page works with no internet."
except Exception:
    PLOTLY_TAG = '<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>'
    plotly_note = ("plotly is not installed, so the page loads Plotly from the internet. "
                   "Run: pip install plotly   to make it work offline.")

def embed_google_fonts(css_url):
    """Download the Latin font files once and embed them as base64, so the page
    keeps its typography with no internet. Raises if anything goes wrong."""
    import base64, urllib.request
    ua = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
    get = lambda u: urllib.request.urlopen(urllib.request.Request(u, headers=ua), timeout=20).read()
    css = get(css_url).decode("utf-8")
    blocks = [b for subset, b in re.findall(r"/\*\s*([\w-]+)\s*\*/\s*(@font-face\s*{[^}]*})", css) if subset == "latin"]
    if not blocks:
        raise ValueError("no Latin font blocks in the response")
    cache, out = {}, []
    for block in blocks:
        m = re.search(r"url\(([^)]+)\)", block)
        src = m.group(1).strip("'\"")
        if src not in cache:
            cache[src] = base64.b64encode(get(src)).decode("ascii")
        out.append(block.replace(m.group(0), "url(data:font/woff2;base64," + cache[src] + ")"))
    return "<style>" + "\n".join(out) + "</style>", len(cache)


try:
    FONTS_TAG, n_files = embed_google_fonts(FONTS_URL)
    fonts_note = f"Fonts embedded ({n_files} files): the page keeps its look offline."
except Exception as exc:
    FONTS_TAG = ('<link rel="preconnect" href="https://fonts.googleapis.com">'
                 '<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>'
                 f'<link href="{FONTS_URL}" rel="stylesheet">')
    fonts_note = (f"Could not embed fonts ({exc.__class__.__name__}), so the page links to "
                  "Google Fonts. It still works offline, with system fonts.")

fs = FULL_STATS
replacements = {
    "__FONTS__": FONTS_TAG,
    "__PLOTLY__": PLOTLY_TAG,
    "__CSS__": PAGE_CSS,
    "__JS__": PAGE_JS,
    "__PAYLOAD__": json.dumps(PAYLOAD, ensure_ascii=False).replace("</", "<\\/"),
    "__N__": str(N_TOTAL),
    "__FAIR__": f"{fs['dimension_means']['Fairness']:.2f}",
    "__PRIDE__": f"{fs['dimension_means']['Pride']:.2f}",
    "__CS__": f"{fs['cs_fairness']:.2f}",
    "__REST__": f"{fs['rest_fairness']:.2f}",
    "__CSP__": f"{fs['cs_p']}",
    "__CORR__": f"{fs['corr_r']:.2f}",
}

html_out = PAGE_HTML
for key, value in replacements.items():
    html_out = html_out.replace(key, value)

leftover = sorted(set(re.findall(r"__[A-Z]+__", html_out.replace(PLOTLY_TAG, "").replace(FONTS_TAG, ""))))
if leftover:
    raise RuntimeError(f"Unfilled placeholders: {leftover}")

out_path = Path(OUTPUT_FILE).resolve()
out_path.write_text(html_out, encoding="utf-8")

print("Cell 10 done.")
print(f"  {plotly_note}")
print(f"  {fonts_note}")
print(f"  Wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

if OPEN_IN_BROWSER:
    try:
        webbrowser.open(out_path.as_uri())
        print("  Opened in your default browser.")
    except Exception as exc:
        print(f"  Could not open a browser automatically ({exc}). Double-click the file instead.")